In [38]:
import numpy as np
import emcee
import corner
import matplotlib.pyplot as plt
np.random.seed(42)
z_bao, bao_val, bao_type = np.loadtxt('data/desi_gaussian_bao_ALL_GCcomb_mean.txt', unpack=True, dtype=str)
bao_cov = np.loadtxt('data/desi_gaussian_bao_ALL_GCcomb_cov.txt')
z_bao = z_bao.astype(float)
bao_val = bao_val.astype(float)

# from GPR
mB_match = np.load("output/mB_baomatch_mean_GPR.npy")
mB_cov   = np.load("output/mB_baomatch_cov_GPR.npy")
h_match  = np.load("output/cc_baomatch_mean_GPR.npy")
h_cov    = np.load("output/cc_baomatch_cov_GPR.npy")

# # from GPR but use minimize
# mB_match = np.load("output/mB_baomatch_mean_GPR_minimize.npy")
# mB_cov   = np.load("output/mB_baomatch_cov_GPR_minimize.npy")
# h_match  = np.load("output/cc_baomatch_mean_GPR.npy")
# h_cov    = np.load("output/cc_baomatch_cov_GPR.npy")

# # from FKM
# mB_match = np.load("output/mB_baomatch_mean_FKM.npy")
# mB_cov   = np.load("output/mB_baomatch_cov_FKM.npy")
# h_match  = np.load("output/cc_baomatch_mean_FKM.npy")
# h_cov    = np.load("output/cc_baomatch_cov_FKM.npy")

mask_use = (bao_type == 'DM_over_rs') | (bao_type == 'DV_over_rs')
# mask_use = (bao_type == 'DM_over_rs')

z_fit       = z_bao[mask_use]
bao_val_fit = bao_val[mask_use]
type_fit    = bao_type[mask_use]

idx_fit = np.where(mask_use)[0]
bao_cov_fit = bao_cov[np.ix_(idx_fit, idx_fit)]
mB_cov_fit  = mB_cov[np.ix_(idx_fit, idx_fit)]
h_cov_fit   = h_cov[np.ix_(idx_fit, idx_fit)]

mB_val_fit = mB_match[mask_use]
h_val_fit  = h_match[mask_use]


In [39]:
import numpy as np
import emcee

def get_eta_th(z, param, model_name):
    if model_name == "linear": return 1.0 + param * z
    if model_name == "fractional": return 1.0 + (param * z) / (1.0 + z)
    if model_name == "logarithmic": return 1.0 + param * np.log(1.0 + z)
    if model_name == "powerlaw": return (1.0 + z)**param

def construct_total_covariance(eta_obs, bao_val, h_val, type_arr, C_mB, C_bao, C_h):
    N, c1 = len(eta_obs), np.log(10) / 5.0
    mask_dm, mask_dv = (type_arr == 'DM_over_rs'), (type_arr == 'DV_over_rs')
    J_mB = np.diag(eta_obs * c1)
    w_b = np.zeros(N)
    w_b[mask_dm], w_b[mask_dv] = -1.0 * eta_obs[mask_dm] / bao_val[mask_dm], -1.5 * eta_obs[mask_dv] / bao_val[mask_dv]
    J_b = np.diag(w_b)
    w_h = np.zeros(N)
    w_h[mask_dv] = -0.5 * eta_obs[mask_dv] / h_val[mask_dv]
    J_h = np.diag(w_h)
    return J_mB @ C_mB @ J_mB + J_b @ C_bao @ J_b + J_h @ C_h @ J_h

def ln_likelihood(theta, z, mB_obs, mB_cov, bao_val, bao_cov, h_val, h_cov, type_arr, model):
    eta_i, MB, rd = theta
    c_l = 299792.458
    DL_s = 10**((mB_obs - MB - 25) / 5.0)
    DM = np.zeros_like(z)
    mask_dm, mask_dv = (type_arr == 'DM_over_rs'), (type_arr == 'DV_over_rs')
    DM[mask_dm] = bao_val[mask_dm] * rd
    DM[mask_dv] = np.sqrt((bao_val[mask_dv]**3 * rd**3 * h_val[mask_dv]) / (z[mask_dv] * c_l))
    eta_obs = DL_s / (DM * (1 + z))
    C = construct_total_covariance(eta_obs, bao_val, h_val, type_arr, mB_cov, bao_cov, h_cov)
    # C += np.eye(len(z)) * 1e-11
    res = eta_obs - get_eta_th(z, eta_i, model)
    try:
        sign, logdet = np.linalg.slogdet(C)
        if sign <= 0: return -np.inf
        return -0.5 * (res @ np.linalg.solve(C, res) + logdet + len(z)*np.log(2*np.pi))
    except np.linalg.LinAlgError: return -np.inf

def ln_prior(theta, prior):
    eta_i, MB, rd = theta
    if not (-0.5 < eta_i < 0.5 and -20.5 < MB < -18.5 and 110 < rd < 180): return -np.inf
    lp = 0.0
    if prior == "P18": lp += -0.5 * ((rd - 147.09) / 0.26)**2
    if prior == "R22": lp += -0.5 * ((MB - (-19.384)) / 0.052)**2
    return lp

def ln_probability(theta, *args):
    lp = ln_prior(theta, args[-1])
    if not np.isfinite(lp): return -np.inf
    return lp + ln_likelihood(theta, *args[:-1])

nwalkers = 32
ndim = 3
p0 = [0.0, -19.38, 147.0] + 1e-3 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(
    nwalkers, ndim, ln_probability, 
    args=(z_fit, mB_val_fit, mB_cov_fit, bao_val_fit, bao_cov_fit, h_val_fit, h_cov_fit, type_fit, "linear", "no prior")
)

print("Running MCMC...")
sampler.run_mcmc(p0, 5000, progress=True)

samples = sampler.get_chain(discard=1000, flat=True)

labels = [r"$\eta_0$", r"$M_B$", r"$r_d$"]
fig = corner.corner(samples, labels=labels, truths=[0.0, -19.384, 147.09], 
                    show_titles=True, quantiles=[0.16, 0.5, 0.84], 
                    title_fmt=".4f")
plt.show()

for i in range(ndim):
    mcmc = np.percentile(samples[:, i], [16, 50, 84])
    q = np.diff(mcmc)
    print(f"{labels[i]}: {mcmc[1]:.4f} (+{q[1]:.4f} / -{q[0]:.4f})")

Running MCMC...


100%|██████████| 5000/5000 [00:09<00:00, 544.46it/s]


$\eta_0$: 0.0048 (+0.0191 / -0.0189)
$M_B$: -19.6054 (+0.1780 / -0.1716)
$r_d$: 153.7229 (+10.9685 / -10.8413)


In [40]:
models = ["linear", "fractional", "logarithmic", "powerlaw"]
priors = ["noprior","P18", "R22"]

data_args = (z_fit, mB_val_fit, mB_cov_fit, bao_val_fit, bao_cov_fit, h_val_fit, h_cov_fit, type_fit)
for model in models:
    for prior in priors:
        fname = f"chain/samples_{model}_{prior}.npy"
        
        print(f"Running: Model={model}, Prior={prior}")
        nwalkers, ndim = 32, 3
        p0 = [0.0, -19.38, 147.0] + 1e-4 * np.random.randn(nwalkers, ndim)
        sampler = emcee.EnsembleSampler(nwalkers, ndim, ln_probability, args=(*data_args, model, prior))
        sampler.run_mcmc(p0, 5000, progress=True)
        
        samples = sampler.get_chain(discard=1000, flat=True)
        for i in range(ndim):
            mcmc = np.percentile(samples[:, i], [16, 50, 84])
            q = np.diff(mcmc)
            print(f"{labels[i]}: {mcmc[1]:.4f} (+{q[1]:.4f} / -{q[0]:.4f})")
        np.save(fname, samples)

Running: Model=linear, Prior=noprior


100%|██████████| 5000/5000 [00:09<00:00, 538.58it/s]


$\eta_0$: 0.0043 (+0.0199 / -0.0188)
$M_B$: -19.6121 (+0.1811 / -0.1733)
$r_d$: 154.2485 (+11.1547 / -11.1028)
Running: Model=linear, Prior=P18


100%|██████████| 5000/5000 [00:09<00:00, 515.54it/s]


$\eta_0$: 0.0029 (+0.0193 / -0.0186)
$M_B$: -19.5097 (+0.0793 / -0.0814)
$r_d$: 147.0918 (+0.2610 / -0.2543)
Running: Model=linear, Prior=R22


...

$\eta_0$: 0.0037 (+0.0432 / -0.0428)
$M_B$: -19.6084 (+0.1910 / -0.1863)
$r_d$: 154.1388 (+11.3574 / -11.2409)
Running: Model=powerlaw, Prior=P18


100%|██████████| 5000/5000 [00:09<00:00, 512.06it/s]


$\eta_0$: -0.0028 (+0.0413 / -0.0426)
$M_B$: -19.4992 (+0.0880 / -0.0918)
$r_d$: 147.0937 (+0.2615 / -0.2558)
Running: Model=powerlaw, Prior=R22


100%|██████████| 5000/5000 [00:09<00:00, 511.43it/s]

$\eta_0$: -0.0170 (+0.0377 / -0.0401)
$M_B$: -19.3991 (+0.0499 / -0.0504)
$r_d$: 143.3770 (+5.6009 / -5.6337)


In [41]:
import numpy as np
from getdist import MCSamples, plots
prior_labels = [ "No Prior", "R22 Prior", "Planck18 Prior"]
prior_colors = ["#5E75AB", "#A84C4D", "#43623D"]

model_params = {
    "linear": [r"\eta_1", r"M_B", r"r_d"],
    "fractional": [r"\eta_2", r"M_B", r"r_d"],
    "logarithmic": [r"\eta_3", r"M_B", r"r_d"],
    "powerlaw": [r"\epsilon", r"M_B", r"r_d"]
}

for model in models:
    mcsamples_list = []
    
    for i, prior in enumerate(priors):
        # Load samples (shape: [N, 3])
        data = np.load(f"chain/samples_{model}_{prior}.npy")
        
        # Create MCSamples object
        samples = MCSamples(
            samples=data,
            names=["eta", "MB", "rd"],
            labels=model_params[model],
            label=prior_labels[i]
        )
        samples.updateSettings({'smooth_scale_1D': 0.5, 'smooth_scale_2D': 0.5})
        mcsamples_list.append(samples)

    g = plots.get_subplot_plotter(subplot_size=3)
    
    # Plot Triangle
    g.triangle_plot(
        mcsamples_list,
        filled=True,
        contour_colors = prior_colors,
        markers={"eta": 0},
        marker_args={"lw": 1},
    )
    
    g.export(f"fig/constraint_{model}_comparison.pdf")

Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
Removed no burn in
